# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf, StorageLevel
import csv
from operator import add

In [2]:
conf = SparkConf().setAppName("Lab4-rdd").setMaster("local[4]")
conf.set("spark.ui.showConsoleProgress", "false")
sc = SparkContext(conf=conf)
sc.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 21:16:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

## Parse CSV records and use patent IDs as keys

Use Python's CSV reader to remove quoting correctly. Remove each file's header and convert patent IDs to integers. Patent values retain every original field after the ID. Repartition the full dataset for the joins; no sampling is used.

In [6]:
citation_header = rddCitations.first()
patent_header = rddPatents.first()
columns = next(csv.reader([patent_header]))
patents = (rddPatents.filter(lambda line: line != patent_header)
           .mapPartitions(csv.reader)
           .map(lambda row: (int(row[0]), tuple(row[1:])))
           .partitionBy(16)
           .persist(StorageLevel.MEMORY_AND_DISK))
citations = (rddCitations.filter(lambda line: line != citation_header)
             .mapPartitions(csv.reader)
             .map(lambda row: (int(row[0]), int(row[1])))
             .partitionBy(16)
             .persist(StorageLevel.MEMORY_AND_DISK))

## Join states and aggregate counts

Build a lookup of US patent IDs to nonempty states. Reverse each citation to key it by the cited patent, then join its state. Re-key by the citing patent and join again to obtain both states. Filter matching states and sum ones for each citing patent.

Left join these counts onto all original patents. Use zero when no qualifying citation exists, preserving every patent and all its original fields.

In [7]:
def augment_patents_rdd(patents, citations):
    states = (patents.filter(lambda item: item[1][3] == "US" and bool(item[1][4]))
              .mapValues(lambda row: row[4]))
    cited_states = citations.map(lambda pair: (pair[1], pair[0])).join(states)
    both_states = (cited_states.map(lambda item: (item[1][0], item[1][1]))
                   .join(states))
    counts = (both_states.filter(lambda item: item[1][0] == item[1][1])
              .mapValues(lambda states: 1)
              .reduceByKey(add))
    return patents.leftOuterJoin(counts).mapValues(
        lambda pair: pair[0] + (pair[1] if pair[1] is not None else 0,)
    )

## Check boundary cases

Use the same example as the DataFrame notebook. Unknown patents, unknown states, foreign patents, different states, and patents without citations must not add to the count.

In [8]:
example_patents = sc.parallelize([
    (1, ("", "", "", "US", "NY")), (2, ("", "", "", "US", "NY")),
    (3, ("", "", "", "US", "CA")), (4, ("", "", "", "US", None)),
    (5, ("", "", "", "US", "")), (6, ("", "", "", "GB", "NY")),
    (7, ("", "", "", "US", "TX"))
], 2)
example_citations = sc.parallelize([
    (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 999),
    (4, 5), (5, 4), (6, 2), (999, 2)
], 2)
actual = augment_patents_rdd(example_patents, example_citations).mapValues(
    lambda row: row[-1]).collectAsMap()
assert actual == {1: 1, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}
print("Boundary-case checks passed.")

Boundary-case checks passed.


## Full dataset result

Cache the augmented records for ranking and validation. Display complete patent records with SAME_STATE appended. Resolve tied counts by ascending patent ID, as in the DataFrame solution.

In [9]:
augmented = augment_patents_rdd(patents, citations).persist(StorageLevel.MEMORY_AND_DISK)
patent_count = patents.count()
citation_count = citations.count()
assert augmented.count() == patent_count
print(f"Patents: {patent_count:,}; citations: {citation_count:,}")
top_ten = augmented.takeOrdered(10, key=lambda item: (-item[1][-1], item[0]))
print(tuple(columns + ["SAME_STATE"]))
for patent, fields in top_ten:
    print((patent,) + fields)

Patents: 2,923,922; citations: 16,522,438


('PATENT', 'GYEAR', 'GDATE', 'APPYEAR', 'COUNTRY', 'POSTATE', 'ASSIGNEE', 'ASSCODE', 'CLAIMS', 'NCLASS', 'CAT', 'SUBCAT', 'CMADE', 'CRECEIVE', 'RATIOCIT', 'GENERAL', 'ORIGINAL', 'FWDAPLAG', 'BCKGTLAG', 'SELFCTUB', 'SELFCTLB', 'SECDUPBD', 'SECDLWBD', 'SAME_STATE')
(5959466, '1999', '14515', '1997', 'US', 'CA', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125)
(5983822, '1999', '14564', '1998', 'US', 'TX', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103)
(6008204, '1999', '14606', '1998', 'US', 'CA', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100)
(5952345, '1999', '14501', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98)
(5958954, '1999', '14515', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397',

## Verify the reference example and totals

The supplied data gives patent 6009554 eight NY citations, whereas the README example says six. Check the observed count of eight and the reference leader with 125. Ties at 90 can change which patents appear in the top ten; both notebooks sort ties by patent ID. Compare the total and squared counts with the DataFrame notebook to check agreement beyond the top ten.

In [10]:
assert augmented.filter(lambda item: item[0] == 6009554).first()[1][-1] == 8
assert (top_ten[0][0], top_ten[0][1][-1]) == (5959466, 125)
summary = augmented.map(lambda item: (item[1][-1], item[1][-1] ** 2)).reduce(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)
print({"total_same_state": summary[0], "sum_squared_counts": summary[1]})
print("Full-dataset checks passed.")

{'total_same_state': 1488330, 'sum_squared_counts': 9103680}
Full-dataset checks passed.


In [12]:
augmented.unpersist()
patents.unpersist()
citations.unpersist()
sc.stop()